# Лабораторная работа. Структуры данных pandas и индексация

**Цель работы:** освоить основные структуры данных библиотеки pandas (`Series`, `DataFrame`, `Index`), научиться загружать данные из файлов, исследовать их, выбирать, фильтровать, сортировать и преобразовывать, а также эффективно работать с большими наборами данных.

**Используемые данные:**
- `sp500.csv` — сведения о 500 компаниях индекса S&P 500 (небольшой CSV, ~80 КБ);
- `sales_1m.csv` — синтетический набор продаж на 1 000 000 строк (генерируется в разделе 13).

**Содержание**
1. Подготовка к работе
2. Структуры данных: Series и DataFrame
3. Объект Index
4. Чтение данных из файлов
5. Первичный осмотр данных
6. Доступ к данным: столбцы, loc, iloc, at, iat, срезы
7. Копирование и ссылки
8. Фильтрация: логические маски, isin(), query()
9. Сортировка
10. Изменение структуры: создание, переименование, удаление столбцов
11. Арифметические операции и выравнивание данных
12. Индексация подробнее: set_index, reindex, MultiIndex
13. Работа с большим датасетом: память, типы, usecols, dtype, chunksize
14. Задания для самостоятельной работы
15. Контрольные вопросы

## 1. Подготовка к работе

**pandas** — библиотека Python для анализа табличных данных. Она построена поверх **NumPy**: значения столбцов хранятся в массивах NumPy (или в расширенных типах pandas), поэтому операции над целыми столбцами выполняются быстро, без циклов Python (векторизация).

Традиционные сокращения при импорте: `pd` для pandas и `np` для NumPy.

In [1]:
import os
import time
import numpy as np
import pandas as pd

print('pandas', pd.__version__)
print('numpy ', np.__version__)

pandas 3.0.2
numpy  2.4.4


Основной набор данных — `sp500.csv` (500 компаний индекса S&P 500, 15 столбцов). Положите файл в папку `../data/` или рядом с notebook.

| Column Name | Description |
| ------------- |:-------------:|
|Symbol|Сокращенное название организации|
|Name|Полное название организации|
|Sector|Сектор экономики|
|Price|Стоимость акции|
|Dividend Yield|Дивидендная доходность|
|Price/Earnings|Цена / прибыль|
|Earnings/Share|Прибыль на акцию|
|Book Value|Балансовая стоимость компании|
|52 week low|52-недельный минимум|
|52 week high|52-недельный максимум|
|Market Cap|Рыночная капитализация|
|EBITDA|**E**arnings **b**efore **i**nterest, **t**axes, **d**epreciation and **a**mortization|
|Price/Sales|Цена / объём продаж|
|Price/Book|Цена / балансовая стоимость|
|SEC Filings|Ссылка *sec.gov*|

In [2]:
# ищем файл в типичных местах
for path in ['../data/sp500.csv', 'sp500.csv', 'data/sp500.csv']:
    if os.path.exists(path):
        DATA_PATH = path
        break
else:
    raise FileNotFoundError('Положите sp500.csv в ../data/ или рядом с notebook')
print('Используется файл:', DATA_PATH)

Используется файл: sp500.csv


## 2. Структуры данных: Series и DataFrame

### Теория
- **Series** — одномерный размеченный массив: значения (`values`) одного типа + метки (`index`). Похож на словарь «метка → значение» и одновременно на массив NumPy.
- **DataFrame** — двумерная таблица: набор столбцов-Series с общим индексом строк. У DataFrame два индекса: `index` (строки) и `columns` (столбцы). Разные столбцы могут иметь разные типы.
- Если индекс не задан, pandas создаёт `RangeIndex` 0, 1, 2, …

Основные конструкторы:
```python
pd.Series(data, index=None, dtype=None, name=None)
pd.DataFrame(data, index=None, columns=None, dtype=None)
```
`data` может быть списком, словарём, массивом NumPy, скаляром, другим Series/DataFrame.

### 2.1 Создание из списков

создаем Series значений из списка целых чисел с явным индексом

In [3]:
s = pd.Series(data=[10, 11, 12, 13, 14],
              index=[1, 2, 3, 5, 7])
s

1    10
2    11
3    12
5    13
7    14
dtype: int64

создаем Series из строковых значений (тип `object`/`str`), индекс создаётся автоматически

In [4]:
s = pd.Series(['Blue', 'Yellow', 'Green'])
s

0      Blue
1    Yellow
2     Green
dtype: str

создаем DataFrame из двумерного списка (список строк) и задаём имена столбцов

In [9]:
df = pd.DataFrame([[10, 11], [20, 21], [30, 31]],
                  columns=['A', 'B'])
df

,A,B
0,10,11
1,20,21
2,30,31


### 

создаем DataFrame из списка объектов Series: каждая Series становится **строкой**

In [16]:
series_1 = pd.Series([70, 90])
series_2 = pd.Series([71, 'A'])
df = pd.DataFrame([ series_1,series_2])
df

,0,1
0,70,90
1,71,A


имена столбцов можно задать и после создания датафрейма

In [17]:
df.columns = ['col_1', 'col_2']
df

,col_1,col_2
0,70,90
1,71,A


### 2.2 Создание из словарей
- Для **Series** ключи словаря становятся индексом, значения — данными.
- Для **DataFrame** ключи становятся **именами столбцов**, а значения (списки или Series) — содержимым столбцов.

In [18]:
s = pd.Series({'Homer': 'Dad',
               'Marge': 'Mom',
               'Bart': 'Son',
               'Lisa': 'Daughter',
               'Maggie': 'Daughter'})
s

Homer          Dad
Marge          Mom
Bart           Son
Lisa      Daughter
Maggie    Daughter
dtype: str

DataFrame из словаря списков

In [19]:
temperatures = {'col_1': [70, 71],
                'col_2': [90, 91]}
pd.DataFrame(temperatures)

,col_1,col_2
0,70,90
1,71,91


DataFrame из словаря Series: каждая Series становится **столбцом**

In [20]:
series_1 = pd.Series([70, 71])
series_2 = pd.Series([90, 91])
df = pd.DataFrame({'col_1': series_1,
                   'col_2': series_2})
df

,col_1,col_2
0,70,90
1,71,91


### 2.3 Создание из массивов NumPy
Часто данные генерируют функциями NumPy:
- `np.arange(start, stop, step)` — числа от **start** до **stop-1** с шагом **step**;
- `np.linspace(start, stop, num)` — `num` равномерно распределённых точек на отрезке, включая концы;
- `np.random.*` — случайные числа. Фиксация `seed` делает результаты воспроизводимыми.

In [21]:
pd.Series(np.arange(15, 25, 2))

0    15
1    17
2    19
3    21
4    23
dtype: int64

In [22]:
pd.Series(np.linspace(0, 9, 5))

0    0.00
1    2.25
2    4.50
3    6.75
4    9.00
dtype: float64

Series из 5 нормально распределенных случайных чисел

In [25]:
np.random.seed(123)
pd.Series(np.random.normal(size=12))

0    -1.085631
1     0.997345
2     0.282978
3    -1.506295
4    -0.578600
5     1.651437
6    -2.426679
7    -0.428913
8     1.265936
9    -0.866740
10   -0.678886
11   -0.094709
dtype: float64

DataFrame 4×3 из двумерного массива NumPy с заданными индексом и столбцами

In [26]:
np.random.seed(123)
df = pd.DataFrame(np.random.normal(size=12).reshape(4, 3),
                  index=['ind_1', 'ind_2', 'ind_3', 'ind_4'],
                  columns=['col_1', 'col_2', 'col_3'])
df

,col_1,col_2,col_3
ind_1,-1.085631,0.997345,0.282978
ind_2,-1.506295,-0.578600,1.651437
ind_3,-2.426679,-0.428913,1.265936
ind_4,-0.866740,-0.678886,-0.094709


### 2.4 Объекты для дальнейших примеров
`Simpsons` — Series с текстовым индексом, `numbers` — Series с целочисленным индексом 25…34 (не совпадающим с позициями 0…9, это важно для понимания разницы `loc` и `iloc`).

In [27]:
Simpsons = pd.Series({'Homer': 120, 'Marge': 60, 'Bart': 35, 'Lisa': 30, 'Maggie': 7})
Simpsons

Homer     120
Marge      60
Bart       35
Lisa       30
Maggie      7
dtype: int64

In [28]:
np.random.seed(123)
numbers = pd.Series(data=np.random.normal(size=10),
                    index=np.arange(25, 35))
numbers

25   -1.085631
26    0.997345
27    0.282978
28   -1.506295
29   -0.578600
30    1.651437
31   -2.426679
32   -0.428913
33    1.265936
34   -0.866740
dtype: float64

## 3. Объект Index

### Теория
**Index** — это объект, хранящий метки строк (`df.index`) или столбцов (`df.columns`). Его свойства:
- **неизменяемость** — нельзя присвоить значение отдельному элементу (`idx[0] = ...` вызовет ошибку), можно только заменить индекс целиком;
- может содержать **повторяющиеся** метки;
- поддерживает **операции над множествами**: `union`, `intersection`, `difference`, `symmetric_difference`;
- обеспечивает **быстрый поиск** по меткам (хеш-таблица), см. раздел 12.

Основные виды индексов:
| Тип | Когда возникает |
|---|---|
| `RangeIndex` | индекс по умолчанию 0, 1, 2, … (хранит только start/stop/step, почти не занимает памяти) |
| `Index` с `dtype=int64` / `float64` / `object` | метки — целые / дробные числа / строки |
| `DatetimeIndex` | метки — даты и время |
| `CategoricalIndex` | метки — категории |
| `MultiIndex` | многоуровневый (иерархический) индекс |

> В pandas 2.0+ классы `Int64Index` и `Float64Index` удалены: вместо них используется обычный `Index` с соответствующим `dtype`.

In [29]:
idx = pd.Index(['a', 'b', 'c', 'd'], name='letters')
print(idx)
print('тип меток:', idx.dtype, '| размер:', idx.size)
try:
    idx[0] = 'z'
except TypeError as e:
    print('Index неизменяем:', e)

Index(['a', 'b', 'c', 'd'], dtype='str', name='letters')
тип меток: str | размер: 4
Index неизменяем: Index does not support mutable operations


## операции над множествами

In [30]:
idx2 = pd.Index(['c', 'd', 'e'])
print('объединение:', idx.union(idx2))
print('пересечение:', idx.intersection(idx2))
print('разность:   ', idx.difference(idx2))

объединение: Index(['a', 'b', 'c', 'd', 'e'], dtype='str')
пересечение: Index(['c', 'd'], dtype='str')
разность:    Index(['a', 'b'], dtype='str')


типы индексов: `RangeIndex` по умолчанию, целочисленный и дробный индекс

In [35]:
df_range = pd.DataFrame(np.arange(10, 15))
df_i64 = pd.DataFrame(np.arange(10, 20), index=np.arange(0, 10))
df_f64 = pd.DataFrame(np.arange(0, 1000, 5), index=np.arange(0.0, 100.0, 0.5))
print(df_range.index)
print(df_i64.index)
print(df_f64.index[:5])


RangeIndex(start=0, stop=5, step=1)
Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype='int64')
Index([0.0, 0.5, 1.0, 1.5, 2.0], dtype='float64')


у Series и DataFrame индекс доступен через `.index`, значения — через `.values`; столбцы DataFrame — тоже объект Index

In [36]:
print(Simpsons.index)
print(Simpsons.values)
print(df.columns)

Index(['Homer', 'Marge', 'Bart', 'Lisa', 'Maggie'], dtype='str')
[120  60  35  30   7]
Index(['col_1', 'col_2', 'col_3'], dtype='str')


индексу и самой Series можно дать имя

In [37]:
Simpsons.name = 'Simpsons weight'
Simpsons.index.name = 'First name'
Simpsons

First name
Homer     120
Marge      60
Bart       35
Lisa       30
Maggie      7
Name: Simpsons weight, dtype: int64

## 4. Чтение данных из файлов

### Теория
pandas читает данные из множества форматов функциями `pd.read_*` и записывает методами `df.to_*`:

| Формат | Чтение | Запись |
|---|---|---|
| CSV / текст | `pd.read_csv()` | `df.to_csv()` |
| Excel | `pd.read_excel()` (нужен пакет `openpyxl`) | `df.to_excel()` |
| JSON | `pd.read_json()` | `df.to_json()` |
| Parquet, SQL, HTML, … | `read_parquet`, `read_sql`, `read_html` | `to_parquet`, `to_sql`, … |

Важные параметры `read_csv()`:
- `sep` — разделитель (`,`, `;`, `\t`);
- `nrows` — прочитать только первые N строк (удобно для «разведки» большого файла);
- `usecols` — прочитать только нужные столбцы;
- `index_col` — столбец, который станет индексом;
- `dtype` — типы столбцов (подробнее в разделе 13);
- `chunksize` — читать по частям, возвращая итератор;
- `encoding`, `decimal`, `parse_dates`, `na_values`- Кодировка текста файла (по умолчанию utf-8), Символ десятичного разделителя (по умолчанию "."), Какие столбцы преобразовать в даты/время, Дополнительные строки, которые считать NaN  соответственно и др.

[Документация по вводу-выводу](http://pandas.pydata.org/pandas-docs/stable/user_guide/io.html)

### 4.1 read_csv()

неверный разделитель: вся строка попадает в один столбец

In [ ]:
pd.read_csv(DATA_PATH, sep=';').head(3)

правильный разделитель + только 3 строки (`nrows`)

In [ ]:
pd.read_csv(DATA_PATH, sep=',', nrows=3)

только нужные столбцы (`usecols`) и столбец `Symbol` в качестве индекса (`index_col`)

In [ ]:
pd.read_csv(DATA_PATH, nrows=3,
            usecols=['Symbol', 'Sector', 'Price', 'Book Value'],
            index_col='Symbol')

`chunksize` возвращает итератор: файл читается порциями по 50 строк

In [ ]:
df_chunk = pd.read_csv(DATA_PATH, chunksize=50,
                       usecols=['Symbol', 'Sector', 'Price', 'Book Value'],
                       index_col='Symbol')
for df_tmp in df_chunk:
    print('DataFrame part:', df_tmp.shape)

загружаем основной датафрейм `sp500`, с которым будем работать дальше

In [ ]:
sp500 = pd.read_csv(DATA_PATH,
                    usecols=['Symbol', 'Sector', 'Price', 'Book Value'],
                    index_col='Symbol')
sp500

### 4.2 read_excel() и read_json()
Сохраним часть данных в Excel и JSON и прочитаем обратно. Для JSON параметр `orient` задаёт структуру файла (`records` — список объектов-строк, `columns`, `index`, `split` и др.).

In [ ]:
sample = sp500.head(10).reset_index()
# Сохраняем DataFrame в файл Excel:
# index=False — не записывать индекс строк как отдельный столбец;
# sheet_name='data' — задать имя листа в книге Excel.
sample.to_excel('sp500_sample.xlsx', index=False, sheet_name='data')

# Сохраняем DataFrame в JSON:
# orient='records' — каждая строка таблицы станет отдельным объектом
# вида {"столбец": значение, ...};
# force_ascii=False — разрешаем не-ASCII символы (например, кириллицу)
# записывать как есть, а не в виде escape-последовательностей \uXXXX.
sample.to_json('sp500_sample.json', orient='records', force_ascii=False)

df_xlsx = pd.read_excel('sp500_sample.xlsx', sheet_name='data', index_col='Symbol')
df_json = pd.read_json('sp500_sample.json', orient='records').set_index('Symbol')
display(df_xlsx.head(3), df_json.head(3))
print('Данные совпадают:', df_xlsx.equals(df_json))

так выглядит JSON в формате `records`:

In [ ]:
print(open('sp500_sample.json', encoding='utf-8').read()[:250])

## 5. Первичный осмотр данных

### Теория
После загрузки данных первым делом нужно понять их структуру:

| Атрибут / метод | Что показывает |
|---|---|
| `.head(n)`, `.tail(n)` | первые / последние n строк (по умолчанию 5) |
| `.shape` | кортеж (число строк, число столбцов) |
| `len(df)`, `.size` | число строк; общее число ячеек |
| `.columns`, `.index` | метки столбцов и строк |
| `.dtypes` | тип каждого столбца |
| `.info()` | сводка: индекс, столбцы, число непустых значений, типы, память |
| `.describe()` | описательные статистики (count, mean, std, min, квартили, max) |
| `.nunique()` | число уникальных значений |

Атрибуты (`shape`, `columns`, `dtypes`) пишутся **без скобок**, методы (`head()`, `info()`) — **со скобками**.

In [ ]:
sp500.head()

In [ ]:
sp500.tail(3)

In [ ]:
print('shape:', sp500.shape)
print('len  :', len(sp500), '| size:', sp500.size)
print('columns:', sp500.columns.tolist())

In [ ]:
sp500.dtypes

In [ ]:
sp500.info()

для числовых столбцов — статистики распределения, для текстовых (`exclude='number'` — все нечисловые столбцы) — count, unique, top, freq

In [ ]:
sp500.describe()

In [ ]:
sp500.describe(exclude='number')

In [ ]:
sp500.nunique()

`nunique()` показывает 13 секторов, хотя в S&P 500 их 10. Посмотрим на значения внимательнее: `repr` показывает лишние пробелы, а `Industries` — опечатка вместо `Industrials`. Это типичная проблема «грязных» данных, её нужно замечать на этапе осмотра.

In [ ]:
print([repr(x) for x in sp500['Sector'].unique()])
sp500['Sector'].value_counts()  # частота встречаемости

исправим: уберём пробелы по краям и заменим опечатку

In [ ]:
sp500['Sector'] = sp500['Sector'].str.strip().replace({'Industries': 'Industrials'})
print('секторов после очистки:', sp500['Sector'].nunique())

### 5.1 Настройки вывода
pandas сокращает вывод больших таблиц. Число отображаемых строк задаёт `pd.options.display.max_rows`. [Все настройки](http://pandas.pydata.org/pandas-docs/stable/user_guide/options.html#available-options)

In [ ]:
print('по умолчанию:', pd.options.display.max_rows)
pd.options.display.max_rows = 10
sp500

## 6. Доступ к данным

### Теория
| Способ | Принцип | Пример | Результат |
|---|---|---|---|
| `df['col']` | столбец по имени | `sp500['Price']` | Series |
| `df[['c1','c2']]` | несколько столбцов | `sp500[['Price','Sector']]` | DataFrame |
| `df.col` | атрибутивный доступ (только для имён без пробелов) | `sp500.Price` | Series |
| `.loc[строки, столбцы]` | по **меткам** | `sp500.loc['MMM', 'Price']` | скаляр/Series/DataFrame |
| `.iloc[строки, столбцы]` | по **позициям** (с 0) | `sp500.iloc[0, 1]` | скаляр/Series/DataFrame |
| `.at[метка, столбец]` | одно значение по меткам (быстрее `loc`) | `sp500.at['MMM','Price']` | скаляр |
| `.iat[i, j]` | одно значение по позициям (быстрее `iloc`) | `sp500.iat[0, 1]` | скаляр |

Правило: одна метка → Series (или скаляр), **список** меток → DataFrame.

Срезы: в `iloc` правая граница **не включается** (как в Python), в `loc` правая граница **включается**.

### 6.1 Столбцы

In [ ]:
print(type(sp500['Sector']))
sp500['Sector'].head()

In [ ]:
print(type(sp500[['Price', 'Book Value']]))
sp500[['Price', 'Book Value']].head()

атрибутивный доступ работает только для имён-идентификаторов: `sp500.Book Value` — синтаксическая ошибка, поэтому используем квадратные скобки

In [ ]:
print(sp500.Price.head(3))
# sp500.Book Value          -> SyntaxError
print(sp500['Book Value'].head(3))

### 6.2 Строки по метке: loc

**Series**: метки 25 и 33 (а не позиции!); метки 0 в индексе `numbers` нет — будет ошибка `KeyError`

In [ ]:
print(numbers.loc[[25, 33]])
try:
    numbers.loc[0]
except KeyError as e:
    print('KeyError: нет метки', e)

**DataFrame**: одна метка → Series, список меток → DataFrame

In [ ]:
print(type(sp500.loc['MMM']))
sp500.loc['MMM']

In [ ]:
sp500.loc[['MMM', 'MSFT']]

### 6.3 Строки по позиции: iloc

**Series**: позиции 5 и −5 (пятая с конца); позиции 10 нет — ошибка `IndexError`

In [ ]:
print(numbers.iloc[[5, -5]])
try:
    numbers.iloc[10]
except IndexError as e:
    print('IndexError:', e)

**DataFrame**: строки на позициях 0 и 2; позицию метки можно узнать через `index.get_loc()`

In [ ]:
sp500.iloc[[0, 2]]

In [ ]:
i1, i2 = sp500.index.get_loc('MMM'), sp500.index.get_loc('A')
print(i1, i2)
sp500.iloc[[i1, i2]]

### 6.4 Скалярные значения: at, iat

In [ ]:
print('at :', sp500.at['MMM', 'Price'])
print('iat:', sp500.iat[0, 1])

### 6.5 Одновременный отбор строк и столбцов

предпочтительно указывать строки и столбцы в **одном** вызове `loc` / `iloc` (а не цепочкой `[...][...]`)

In [ ]:
sp500.loc[['ABT', 'ZTS'], ['Sector', 'Price']]

In [ ]:
print(sp500.index.get_loc('ABT'), sp500.index.get_loc('ZTS'))
sp500.iloc[[1, 499], [0, 1]]

### 6.6 Срезы
Срез задаётся правилом `[начало : конец : шаг]`:
- в `iloc` правая граница **не включается**, в `loc` — **включается**;
- шаг и позиция могут быть отрицательными (отсчёт с конца);
- нумерация позиций — с нуля.

In [ ]:
numbers.iloc[1:6]     # позиции 1..5

In [ ]:
numbers.iloc[1:6:2]   # позиции 1, 3, 5

In [ ]:
numbers.iloc[:6]

In [ ]:
numbers.iloc[3:]

In [ ]:
numbers.iloc[5::-1]   # в обратном порядке, начиная с позиции 5

In [ ]:
numbers.iloc[-4:]     # 4 последних

DataFrame: срезы по позициям и по меткам

In [ ]:
sp500.iloc[4::-1]

In [ ]:
sp500.loc['ABT':'ACN']   # правая метка включена

### 6.7 Случайная подвыборка и транспонирование

In [ ]:
sp500.sample(n=3, random_state=1)

In [ ]:
sp500.sample(frac=0.01, replace=True, random_state=777)   # с возвращением

In [ ]:
sp500.head().T

## 7. Копирование и ссылки

### Теория
Выборка данных может быть **копией** (новые данные) или **представлением** (view — ссылкой на исходные данные). Раньше изменение выборки иногда меняло исходный объект и вызывало предупреждение `SettingWithCopyWarning`.

Начиная с **pandas 3.0** по умолчанию действует режим **Copy-on-Write**: любой результат выборки ведёт себя как копия. Изменения выборки **никогда не меняют исходный объект**, а данные физически копируются только в момент изменения.

Практические правила:
- чтобы изменить исходный датафрейм, присваивайте через `df.loc[строки, столбец] = значение`;
- если выборку планируется изменять и хранить, явно вызывайте `.copy()`;
- не используйте цепочечное присваивание `df[...][...] = ...` — в pandas 3 оно не изменяет `df`.

In [ ]:
n = numbers.iloc[1:5]       # выборка
n.loc[:] = 0                # изменяем выборку
print(n)
print('\nИсходный numbers не изменился:\n', numbers.iloc[:6])

изменение исходного объекта — только через `loc`/`iloc` на нём самом; для экспериментов работаем с копией

In [ ]:
numbers_copy = numbers.copy()
numbers_copy.iloc[1:5] = 0
display(pd.DataFrame({'numbers': numbers, 'numbers_copy': numbers_copy}).head(6))

## 8. Фильтрация: логические маски, isin(), query()

### Теория
**Логическая маска** — Series из `True`/`False` той же длины, что и данные. Передав её в `[]` или `loc`, получаем только строки с `True`.

- Сравнения: `==`, `!=`, `>`, `<`, `>=`, `<=`.
- Логические операторы: `&` (И), `|` (ИЛИ), `~` (НЕ). Ключевые слова `and`/`or`/`not` для Series **не работают**.
- **Каждое условие — в скобках**: `(x > 0) & (x < 1)`. Без скобок `x > 0 & x < 1` вызовет ошибку из‑за приоритета операторов.
- `.isin(список)` — проверка принадлежности множеству значений (замена цепочки `==` через `|`).
- `.query("строка условия")` — компактная запись фильтра; внешние переменные подставляются через `@имя`.
- `.where(условие, other)` — сохраняет форму, заменяя неподходящие значения на `NaN` или `other`.
- `.all()`, `.any()`, `.sum()` для масок: все ли / хотя бы одно / сколько `True`.

### 8.1 Series

In [ ]:
logical_results = (numbers > 0) & (numbers < 1)
logical_results

In [ ]:
numbers[logical_results]

In [ ]:
numbers.where((numbers > 0) & (numbers < 1), other=-1)

In [ ]:
print('все >= 0 :', (numbers >= 0).all())
print('есть < 2 :', (numbers < 2).any())
print('сколько < 1:', (numbers < 1).sum())

### 8.2 DataFrame

In [ ]:
sp500[sp500.Price < 100].head()

строки, где 6 < Price < 10, только столбец Price

In [ ]:
sp500.loc[(sp500['Price'] < 10) & (sp500['Price'] > 6), 'Price']

сектор Health Care и Price ≥ 100

In [ ]:
sp500.loc[(sp500.Sector == 'Health Care') & (sp500.Price >= 100), ['Price', 'Sector']].head()

### 8.3 isin()

In [ ]:
mask = sp500.Sector.isin(['Information Technology', 'Financials'])
print('отобрано строк:', mask.sum())
sp500[mask].head()

отрицание `~`: все секторы, **кроме** перечисленных

In [ ]:
sp500[~mask].Sector.unique()

### 8.4 query()
Тот же фильтр, что в 8.2, но короче. Внешняя переменная — через `@`.

In [ ]:
min_price = 100
sp500.query("Sector == 'Health Care' and Price >= @min_price")[['Price', 'Sector']].head()

имена столбцов с пробелами в `query` заключаются в обратные кавычки

In [ ]:
sp500.query('`Book Value` > Price').head()

## 9. Сортировка

### Теория
- `sort_values(by, ascending=True)` — по значениям одного или нескольких столбцов; для нескольких столбцов `ascending` можно задать списком.
- `sort_index(axis=0)` — по меткам строк; `axis=1` — по именам столбцов.
- `nlargest(n, col)` / `nsmallest(n, col)` — n наибольших / наименьших значений (быстрее полной сортировки).
- Методы возвращают **новый** объект; исходный не меняется.

In [ ]:
sp500.sort_index().head()

In [ ]:
sp500.sort_index(axis=1).head()

In [ ]:
sp500.sort_values(by='Price', ascending=False).head()

сортировка по нескольким столбцам: сектор по алфавиту, внутри сектора — по убыванию цены

In [ ]:
sp500.sort_values(by=['Sector', 'Price'], ascending=[True, False]).head(8)

In [ ]:
sp500.nlargest(5, 'Price')

In [ ]:
sp500.nsmallest(5, 'Price')

## 10. Изменение структуры: создание, переименование, удаление столбцов

### Теория
**Создание столбцов**
- `df['new'] = выражение` — добавить в конец (изменяет `df`);
- `df.insert(позиция, 'new', значения)` — вставить на нужную позицию (изменяет `df`);
- `df.assign(new=выражение, ...)` — вернуть **новый** датафрейм; можно использовать `lambda` для ссылки на только что созданные столбцы.

**Переименование**
- `df.rename(columns={'старое': 'новое'})` / `rename(index={...})` — возвращает новый объект;
- `df.columns = [...]` — заменить все имена сразу.

**Удаление**
- `df.drop(columns=[...])` / `df.drop(index=[...])` — возвращает новый объект (то же, что `axis=1` / `axis=0`);
- `del df['col']` — удалить столбец на месте;
- `df.pop('col')` — удалить столбец на месте и вернуть его как Series.

Параметр `inplace=True` изменяет объект на месте, но его использование не рекомендуется: присваивание результата (`df = df.drop(...)`) понятнее.

### 10.1 Создание

In [ ]:
sp500_copy = sp500.copy()
sp500_copy['RoundedPrice'] = sp500_copy.Price.round()
sp500_copy.head(3)

In [ ]:
sp500_copy = sp500.copy()
sp500_copy.insert(1, 'RoundedPrice', sp500_copy.Price.round())
sp500_copy.head(3)

`assign` создаёт два столбца, второй вычисляется из первого через `lambda`

In [ ]:
sp500.assign(Rounded_Price=sp500.Price.round(),
             R_BookValue_Price=lambda x: x['Book Value'] / x['Rounded_Price']).head(3)

### 10.2 Переименование

In [ ]:
sp500_copy = sp500.rename(columns={'Book Value': 'BookValue'})
print('исходный:', sp500.columns.tolist())
print('копия   :', sp500_copy.columns.tolist())

In [ ]:
sp500.rename(index={'MMM': '3M'}).head(2)

### 10.3 Удаление

`drop` — новый объект без столбца `Sector`; исходный не изменился

In [ ]:
sp500_copy = sp500.copy()
display(sp500_copy.drop(columns=['Sector']).head(3))
print(sp500_copy.columns.tolist())

удаление строк по меткам

In [ ]:
sp500.iloc[:5].drop(index=['ABT', 'ACN'])

`del` и `pop` удаляют на месте; `pop` ещё и возвращает удалённый столбец. Для Series работают аналогично (по метке элемента).

In [ ]:
sp500_copy = sp500.copy()
del sp500_copy['Price']
popped_column = sp500_copy.pop('Sector')
display(sp500_copy.head(3))
popped_column.head(3)

In [ ]:
Simpsons_copy = Simpsons.copy()
del Simpsons_copy['Maggie']
Simpsons_copy

## 11. Арифметические операции и выравнивание данных

### Теория
Арифметические операции (`+ - * / // % **`) в pandas **векторизованы** — применяются ко всем элементам сразу:
- столбец и скаляр: `df['Price'] * 1.1`;
- столбец и столбец: `df['Price'] / df['Book Value']`;
- DataFrame и Series: по умолчанию Series сопоставляется со **столбцами**.

**Выравнивание по индексу.** При операции над двумя объектами pandas сначала сопоставляет их **метки**, а не позиции. Для меток, которых нет хотя бы в одном объекте, результат — `NaN`. Чтобы заменить отсутствующие значения, используют методы `add`, `sub`, `mul`, `div` с параметром `fill_value`.

Агрегирующие функции: `sum()`, `mean()`, `median()`, `min()`, `max()`, `std()`, `count()`.

### 11.1 Векторная арифметика

In [ ]:
ar = sp500.head().copy()
ar['Price_x2'] = ar['Price'] * 2
ar['Price_minus_BV'] = ar['Price'] - ar['Book Value']
ar['Price_to_Book'] = (ar['Price'] / ar['Book Value']).round(2)
ar

операция DataFrame − Series: из каждого столбца вычитается его среднее

In [ ]:
num = sp500[['Price', 'Book Value']]
(num - num.mean()).head()

In [ ]:
print('сумма   :', round(sp500['Price'].sum(), 2))
print('среднее :', round(sp500['Price'].mean(), 2))
print('медиана :', sp500['Price'].median())
print('макс    :', sp500['Price'].max())

### 11.2 Выравнивание Series

In [ ]:
s_1 = pd.Series([77, 33, 11], index=['a', 'b', 'f'])
s_2 = pd.Series([11, 5, 6], index=['c', 'b', 'a'])
display(pd.DataFrame({'s_1': s_1, 's_2': s_2, 's_1 + s_2': s_1 + s_2,
                      'add(fill_value=0)': s_1.add(s_2, fill_value=0)}))

метки не обязательно уникальны: 3 метки 'a' и 2 метки 'a' дают 3×2 = 6 меток 'a' в результате

In [ ]:
s_1 = pd.Series([77, 33, 15, 3], index=['a', 'a', 'a', 'd'])
s_2 = pd.Series([11, 5, 6], index=['c', 'a', 'a'])
s_2 + s_1

### 11.3 Выравнивание DataFrame
Выравнивание происходит и по строкам, и по столбцам.

In [ ]:
sp500_part_1 = sp500.iloc[0:5, 0:2].copy()
sp500_part_2 = sp500.iloc[2:7, 1:3].copy()
display(sp500_part_1, sp500_part_2)
sp500_part_1 + sp500_part_2

выравнивание происходит и при создании DataFrame из Series с разными индексами

In [ ]:
df = pd.DataFrame({'col_1': pd.Series([70, 90]),
                   'col_2': pd.Series([71, 91]),
                   'col_3': pd.Series([85, 87], index=[1, 2])})
df

## 12. Индексация подробнее

### Теория
Индекс — не просто «номера строк». Хорошо выбранный индекс:
- ускоряет поиск: `loc` по индексу использует хеш-таблицу, а фильтр по столбцу `df[df.key == x]` проверяет все строки;
- позволяет выравнивать данные при арифметике и объединении таблиц.

Операции с индексом:
- `set_index(col)` — сделать столбец индексом; `reset_index()` — вернуть индекс в столбцы;
- `reindex(index=..., columns=...)` — привести к заданному набору меток (новые метки получают `NaN` или `fill_value`);
- **MultiIndex** — иерархический индекс из нескольких уровней; выбор по уровню — `xs(метка, level=...)` или `loc[(уровень0, уровень1)]`.

### 12.1 Мотивация: скорость доступа

In [ ]:
np.random.seed(123)
df = pd.DataFrame({'value': np.random.random(10000), 'key': range(100, 10100)})
df_with_index = df.set_index('key')
df_with_index.head()

In [ ]:
%timeit df[df.key == 10099]

In [ ]:
%timeit df_with_index.loc[10099]

<center><font color='green'><b>Вывод: поиск по индексу выполняется в десятки раз быстрее, чем фильтрация по столбцу.</b></font></center>

### 12.2 reset_index и set_index

In [ ]:
index_moved_to_col = sp500.reset_index()
index_moved_to_col.head(3)

In [ ]:
index_moved_to_col.set_index('Sector').head(3)

### 12.3 Переиндексация (reindex)

`loc` с несуществующей меткой вызывает ошибку, а `reindex` создаёт строку с `NaN`

In [ ]:
try:
    sp500.loc[['MMM', 'ABBV', 'NEW VALUE']]
except KeyError as e:
    print('KeyError:', e)
sp500.reindex(index=['MMM', 'ABBV', 'NEW VALUE'])

In [ ]:
sp500.reindex(columns=['Price', 'Book Value', 'NewCol'], fill_value=0).head(3)

### 12.4 Операции над множествами индексов
Какие компании попали в обе случайные выборки, в любую из них, только в первую?

In [ ]:
part_1 = sp500.sample(100, random_state=333)
part_2 = sp500.sample(100, random_state=777)
print('объединение:', len(part_1.index.union(part_2.index)))
print('пересечение:', len(part_1.index.intersection(part_2.index)))
print('разность   :', len(part_1.index.difference(part_2.index)))

### 12.5 Иерархический индекс (MultiIndex)

In [ ]:
multi_fi = sp500.reset_index().set_index(['Sector', 'Symbol']).sort_index()
print(type(multi_fi.index), '| уровней:', multi_fi.index.nlevels)
multi_fi.head()

каждый уровень — отдельный Index

In [ ]:
multi_fi.index.levels[0]

In [ ]:
multi_fi.reorder_levels([1, 0], axis=0).head()

все акции сектора Industrials (уровень 0 исчезает из результата)

In [ ]:
multi_fi.xs('Industrials').head()

отбор по уровню 1 (Symbol)

In [ ]:
multi_fi.xs('ALLE', level=1)

комбинация уровней через кортеж

In [ ]:
multi_fi.loc[('Industrials', 'UPS')]

## 13. Работа с большим датасетом: память, типы данных, usecols, dtype, chunksize

### Теория
На маленьких таблицах эффективность незаметна, но при сотнях тысяч и миллионах строк данные могут занять гигабайты памяти.

**Сколько занимает DataFrame:** примерно `число строк × сумма размеров значений всех столбцов`.

| Тип | Байт на значение | Диапазон / комментарий |
|---|---|---|
| `int8` / `int16` / `int32` / `int64` | 1 / 2 / 4 / 8 | int8: −128…127 |
| `float32` / `float64` | 4 / 8 | float32 ≈ 7 значащих цифр |
| `bool` | 1 | |
| `datetime64` | 8 | |
| `object` / `str` (строки) | десятки байт | каждая строка — отдельный объект Python |
| `category` | 1–2 (коды) + словарь значений | для повторяющихся строк |

**Способы уменьшить память:**
1. `memory_usage(deep=True)` / `info(memory_usage='deep')` — измерить реальный расход (без `deep` строки не учитываются).
2. **Подобрать типы**: `astype()`, `pd.to_numeric(..., downcast=...)`, `category` для строк с малым числом уникальных значений.
3. **`usecols`** — не загружать ненужные столбцы.
4. **`dtype={...}`** — задать компактные типы уже при чтении.
5. **Фильтрация строк и выбор столбцов** — уменьшают соответственно число строк и число столбцов в формуле выше. После фильтрации делайте `.copy()` и удаляйте большой объект `del df`.
6. **`chunksize`** — обрабатывать файл по частям, если он не помещается в память.

### 13.1 Генерация датасета на 1 000 000 строк

In [ ]:
N = 1_000_000
rng = np.random.default_rng(0)
big = pd.DataFrame({
    'order_id': np.arange(N),
    'date': pd.date_range('2023-01-01', periods=N, freq='min').strftime('%Y-%m-%d'),
    'region': rng.choice(['Центр', 'Юг', 'Север', 'Урал', 'Сибирь'], N),
    'category': rng.choice(['Электроника', 'Одежда', 'Продукты', 'Книги'], N),
    'qty': rng.integers(1, 20, N),
    'price': rng.uniform(10, 5000, N).round(2),
    'comment': rng.choice(['ok', 'возврат', 'доставка задержана', ''], N)})
big.to_csv('sales_1m.csv', index=False)
del big
print(f'Размер файла: {os.path.getsize("sales_1m.csv") / 1024**2:.1f} МБ')

### 13.2 Полная загрузка и измерение памяти

In [ ]:
def mb(d):
    """Объём DataFrame в памяти, МБ (с учётом строк)."""
    return d.memory_usage(deep=True).sum() / 1024**2

t = time.time()
sales = pd.read_csv('sales_1m.csv')
t_full = time.time() - t
print(f'Чтение: {t_full:.2f} c, shape = {sales.shape}')
sales.info(memory_usage='deep')

память по столбцам: строковые столбцы занимают больше всего

In [ ]:
mem = (sales.memory_usage(deep=True) / 1024**2).round(2)
print(mem)
print(f'Итого: {mem.sum():.1f} МБ')

### 13.3 Оптимизация типов данных

In [ ]:
opt = sales.copy()
opt['order_id'] = pd.to_numeric(opt['order_id'], downcast='integer')
opt['qty'] = opt['qty'].astype('int8')
opt['price'] = opt['price'].astype('float32')
for col in ['region', 'category', 'comment']:
    opt[col] = opt[col].astype('category')
opt['date'] = pd.to_datetime(opt['date'])
print(f'До: {mb(sales):.1f} МБ, после: {mb(opt):.1f} МБ, экономия в {mb(sales) / mb(opt):.1f} раз')
opt.dtypes

почему `category` так эффективен: хранится словарь из нескольких уникальных значений и маленький код для каждой строки

In [ ]:
print(opt['region'].cat.categories)
print(opt['region'].cat.codes.dtype, opt['region'].cat.codes.head().tolist())

### 13.4 usecols и dtype при чтении

In [ ]:
t = time.time()
part = pd.read_csv('sales_1m.csv', usecols=['region', 'qty', 'price'],
                   dtype={'region': 'category', 'qty': 'int8', 'price': 'float32'})
t_part = time.time() - t
print(f'Все столбцы     : {mb(sales):7.1f} МБ, {t_full:.2f} c')
print(f'usecols + dtype : {mb(part):7.1f} МБ, {t_part:.2f} c')
part.dtypes

### 13.5 Почему фильтрация и выбор столбцов уменьшают объём данных

In [ ]:
cond = (sales['region'] == 'Юг') & (sales['price'] > 4000)
cols = sales[['region', 'price']]
filtered = sales[cond].copy()
both = sales.loc[cond, ['region', 'price']].copy()
report = pd.DataFrame({
    'строк': [len(sales), len(cols), len(filtered), len(both)],
    'столбцов': [sales.shape[1], cols.shape[1], filtered.shape[1], both.shape[1]],
    'МБ': [mb(sales), mb(cols), mb(filtered), mb(both)]},
    index=['весь датасет', 'только 2 столбца', 'фильтр строк', 'фильтр + 2 столбца']).round(2)
report['доля памяти, %'] = (report['МБ'] / report['МБ'].iloc[0] * 100).round(1)
report

### 13.6 chunksize: обработка файла по частям
В памяти одновременно находится только один кусок; результаты по кускам объединяются.

In [ ]:
total = None
max_chunk_mb = 0
for chunk in pd.read_csv('sales_1m.csv', usecols=['region', 'qty', 'price'],
                         dtype={'region': 'category', 'qty': 'int8', 'price': 'float32'},
                         chunksize=100_000):
    max_chunk_mb = max(max_chunk_mb, mb(chunk))
    chunk['revenue'] = chunk['qty'] * chunk['price']
    part_sum = chunk.groupby('region', observed=True)['revenue'].sum() #groupby группирует по региону и будет подробнее рассмотрен в следующей работе
    total = part_sum if total is None else total.add(part_sum, fill_value=0)
print(f'Макс. память одного куска: {max_chunk_mb:.1f} МБ')
total.sort_values(ascending=False).round(0).astype('int64')

проверка: результат по частям совпадает с расчётом на всём датасете

In [ ]:
check = (sales['qty'] * sales['price']).groupby(sales['region']).sum()
print(np.allclose(check.sort_index().values, total.sort_index().values, rtol=1e-4))
del sales, opt, part, cols, filtered, both

### Выводы
1. Строковые столбцы `object`/`str` — главный потребитель памяти; `category` сокращает их в десятки раз.
2. `usecols` и `dtype` уменьшают память и ускоряют чтение уже на этапе загрузки.
3. Выбор столбцов и фильтрация строк уменьшают объём данных пропорционально оставшейся части.
4. `chunksize` позволяет обрабатывать файлы больше оперативной памяти.

## 14. Задания для самостоятельной работы

Все задания выполняются только средствами, разобранными в этой работе; у каждого указаны разделы, на которые оно опирается. Решения записывайте в ячейки под условием. Ответы на вопросы «объясните» оформляйте в markdown-ячейках.

Данные: `sp500.csv` (переменная `DATA_PATH`) и `sales_1m.csv` (создаётся в разделе 13).

### Задание 1. Создание структур данных и Index
*Разделы 2–3*
1. Создайте Series `grades` с оценками пяти студентов из **словаря** (имя → оценка). Задайте Series имя `'grade'`, а индексу — имя `'student'`.
2. Создайте DataFrame `students` из **словаря списков** со столбцами `group` и `age`, указав в качестве `index` имена тех же студентов.
3. Создайте DataFrame `scores` из **массива NumPy**: зафиксируйте `np.random.seed(1)`, сгенерируйте `np.random.normal(size=15)`, преобразуйте в форму 5×3 через `reshape`, задайте столбцы `test_1`, `test_2`, `test_3` и тот же индекс.
4. Создайте Series `weeks` из `np.arange(1, 11)` и Series `points` из `np.linspace(0, 100, 5)`.
5. Создайте `pd.Index` из трёх имён (два из них должны быть в `students`, одно — нет). Найдите объединение, пересечение и разность с `students.index`.
6. Попробуйте заменить первый элемент индекса `students.index[0] = 'X'`, перехватив ошибку через `try/except`. Объясните, почему возникает ошибка.

In [19]:
students_map = {'Vladik' : 5, 'Popandupalo Alex' : 2, 'Anderson Erik' : 4, 'Gosha' : 3, 'Verity' : 67}
grades = pd.Series(students_map,name = 'grade')
grades.index.name = 'student'
grades

student
Vladik               5
Popandupalo Alex     2
Anderson Erik        4
Gosha                3
Verity              67
Name: grade, dtype: int64

In [25]:
data_students = [['222',19],['222',19],['222',19],['222',18],['67',1984]]
students = pd.DataFrame(data = data_students, index = students_map, columns=['group','age'])
students

,group,age
Vladik,222,19
Popandupalo Alex,222,19
Anderson Erik,222,19
Gosha,222,18
Verity,67,1984


In [30]:
np.random.seed(1)
scores = pd.DataFrame(np.random.normal(size=15).reshape(5,3),index = students_map ,columns = ['test1', 'test2', 'test3'] )
scores

,test1,test2,test3
Vladik,1.624345,-0.611756,-0.528172
Popandupalo Alex,-1.072969,0.865408,-2.301539
Anderson Erik,1.744812,-0.761207,0.319039
Gosha,-0.249370,1.462108,-2.060141
Verity,-0.322417,-0.384054,1.133769


In [32]:
#4. Создайте Series `weeks` из `np.arange(1, 11)` и Series `points` из `np.linspace(0, 100, 5)`.
weeks = pd.Series(np.arange(1,11))
weeks

0     1
1     2
2     3
3     4
4     5
5     6
6     7
7     8
8     9
9    10
dtype: int64

In [33]:
# 4 (продолжение)
points = pd.Series(np.linspace(0, 100, 5))
points

0      0.0
1     25.0
2     50.0
3     75.0
4    100.0
dtype: float64

In [34]:
# 5
idx_new = pd.Index(['Vladik', 'Gosha', 'Oleg'])
print('Объединение:', students.index.union(idx_new))
print('Пересечение:', students.index.intersection(idx_new))
print('Разность:   ', students.index.difference(idx_new))

Объединение: Index(['Anderson Erik', 'Gosha', 'Oleg', 'Popandupalo Alex', 'Verity',
       'Vladik'],
      dtype='object')
Пересечение: Index(['Vladik', 'Gosha'], dtype='object')
Разность:    Index(['Anderson Erik', 'Popandupalo Alex', 'Verity'], dtype='object')


In [35]:
# 6
try:
    students.index[0] = 'X'
except TypeError as e:
    print('Ошибка:', e)

Ошибка: Index does not support mutable operations


**Объяснение к п. 6:** Ошибка `TypeError` возникает потому, что объект `Index` в pandas является **неизменяемым**. Нельзя заменить отдельный элемент индекса по позиции — можно только заменить весь индекс целиком.

### Задание 2. Чтение файлов и первичный осмотр
*Разделы 4–5*
1. Прочитайте первые 5 строк `sp500.csv` (`nrows`), а затем весь файл в DataFrame `sp` только со столбцами `Symbol`, `Name`, `Sector`, `Price`, `Dividend Yield`, `Market Cap`, сделав `Symbol` индексом.
2. Выведите `head(3)`, `tail(3)`, `shape`, `columns`, `dtypes`.
3. Выведите `info()` и по столбцу `Non-Null Count` определите, в каких столбцах есть пропуски и сколько их (сравните с `len(sp)`).
4. Выведите `describe()`. Найдите в нём медиану цены (строка `50%`) и проверьте её методом `median()`.
5. Очистите названия секторов так, как показано в разделе 5 (`str.strip()` и `replace()`), и выведите число уникальных секторов `nunique()`.
6. Сохраните первые 20 строк в Excel и JSON (`orient='records'`), прочитайте обратно функциями `read_excel()` и `read_json()` и проверьте совпадение методом `equals()`.

In [36]:
# 1
display(pd.read_csv(DATA_PATH, nrows=5))

sp = pd.read_csv(DATA_PATH,
                 usecols=['Symbol', 'Name', 'Sector', 'Price', 'Dividend Yield', 'Market Cap'],
                 index_col='Symbol')
sp.head(3)

,Symbol,Name,Sector,Price,Dividend Yield,Price/Earnings,Earnings/Share,Book Value,52 week low,52 week high,Market Cap,EBITDA,Price/Sales,Price/Book,SEC Filings
0,MMM,3M Co.,Industrials,141.14,2.12,20.33,6.900,26.668,107.15,143.37,92.345,8.121,2.95,5.26,http://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=MMM
1,ABT,Abbott Laboratories,Health Care,39.60,1.82,25.93,1.529,15.573,32.70,40.49,59.477,4.359,2.74,2.55,http://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=ABT
2,ABBV,AbbVie Inc.,Health Care,53.95,3.02,20.87,2.570,2.954,40.10,54.78,85.784,7.190,4.48,18.16,http://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=ABBV
3,ACN,Accenture,Information Technology,79.79,2.34,19.53,4.068,8.326,69.00,85.88,50.513,4.423,1.75,9.54,http://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=ACN
4,ACE,ACE Limited,Financials,102.91,2.21,10.00,10.293,86.897,84.73,104.07,34.753,4.275,1.79,1.18,http://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=ACE


,Name,Sector,Price,Dividend Yield,Market Cap
Symbol,,,,,
MMM,3M Co.,Industrials,141.14,2.12,92.345
ABT,Abbott Laboratories,Health Care,39.60,1.82,59.477
ABBV,AbbVie Inc.,Health Care,53.95,3.02,85.784


In [37]:
# 2
display(sp.head(3))
display(sp.tail(3))
print('shape:', sp.shape)
print('columns:', sp.columns.tolist())
print('\ndtypes:\n', sp.dtypes)

,Name,Sector,Price,Dividend Yield,Market Cap
Symbol,,,,,
MMM,3M Co.,Industrials,141.14,2.12,92.345
ABT,Abbott Laboratories,Health Care,39.60,1.82,59.477
ABBV,AbbVie Inc.,Health Care,53.95,3.02,85.784


,Name,Sector,Price,Dividend Yield,Market Cap
Symbol,,,,,
ZMH,Zimmer Holdings,Health Care,101.84,0.81,17.091
ZION,Zions Bancorp,Financials,28.43,0.56,5.257
ZTS,Zoetis Inc,Health Care,30.53,0.90,15.296


shape: (500, 5)
columns: ['Name', 'Sector', 'Price', 'Dividend Yield', 'Market Cap']

dtypes:
 Name               object
Sector             object
Price             float64
Dividend Yield    float64
Market Cap        float64
dtype: object


In [38]:
# 3
sp.info()
print('\nВсего строк:', len(sp))
print('Пропусков по столбцам:\n', len(sp) - sp.count())

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, MMM to ZTS
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            500 non-null    object 
 1   Sector          500 non-null    object 
 2   Price           500 non-null    float64
 3   Dividend Yield  424 non-null    float64
 4   Market Cap      496 non-null    float64
dtypes: float64(3), object(2)
memory usage: 23.4+ KB

Всего строк: 500
Пропусков по столбцам:
 Name               0
Sector             0
Price              0
Dividend Yield    76
Market Cap         4
dtype: int64


In [39]:
# 4
display(sp.describe())
print('Медиана Price через median():', sp['Price'].median())

,Price,Dividend Yield,Market Cap
count,500.00000,424.000000,496.000000
mean,77.36860,2.131439,35.348525
std,87.59694,1.321017,54.839169
min,0.00000,0.040000,0.208500
25%,38.74500,1.260000,9.859250
50%,58.35500,1.890000,17.444500
75%,86.68000,2.820000,34.653250
max,1197.12000,10.640000,529.000000


Медиана Price через median(): 58.355000000000004


In [40]:
# 5
sp['Sector'] = sp['Sector'].str.strip().replace({'Industries': 'Industrials'})
print('Уникальных секторов после очистки:', sp['Sector'].nunique())

Уникальных секторов после очистки: 10


In [41]:
# 6
sample20 = sp.head(20).reset_index()
sample20.to_excel('sp500_20.xlsx', index=False, sheet_name='data')
sample20.to_json('sp500_20.json', orient='records', force_ascii=False)

df_xlsx = pd.read_excel('sp500_20.xlsx', sheet_name='data', index_col='Symbol')
df_json = pd.read_json('sp500_20.json', orient='records').set_index('Symbol')

print('Данные совпадают:', df_xlsx.equals(df_json))

Данные совпадают: True


### Задание 3. Доступ к данным
*Раздел 6*
На датафрейме `sp` из задания 2:
1. Выведите цену компании `MSFT` четырьмя способами: `loc`, `at`, `iloc` и `iat` (позицию строки найдите через `index.get_loc()`, позицию столбца — через `sp.columns.get_loc()`: столбцы тоже являются объектом Index, раздел 3).
2. С помощью `type()` определите тип результата для `sp.loc['MSFT', 'Price']`, `sp.loc['MSFT']`, `sp.loc[['MSFT'], ['Price']]`. Сопоставьте с таблицей раздела 6.
3. С помощью `iloc` выведите строки с 10-й по 19-ю позицию и столбцы с 1-го по 3-й.
4. С помощью `loc` выведите строки от метки `ABT` до метки `ACN` включительно и только столбцы `Name` и `Price`.
5. Выведите последние 5 строк в обратном порядке срезом `iloc` с отрицательным шагом.
6. Объясните, почему `sp.iloc[0:3]` возвращает 3 строки, а `sp.loc['ABT':'ACN']` включает строку `ACN`.

In [42]:
# 1
row_i = sp.index.get_loc('MSFT')
col_i = sp.columns.get_loc('Price')

print('loc :', sp.loc['MSFT', 'Price'])
print('at  :', sp.at['MSFT', 'Price'])
print('iloc:', sp.iloc[row_i, col_i])
print('iat :', sp.iat[row_i, col_i])

loc : 40.12
at  : 40.12
iloc: 40.12
iat : 40.12


In [43]:
# 2
print("sp.loc['MSFT', 'Price']     ->", type(sp.loc['MSFT', 'Price']))
print("sp.loc['MSFT']              ->", type(sp.loc['MSFT']))
print("sp.loc[['MSFT'], ['Price']] ->", type(sp.loc[['MSFT'], ['Price']]))

sp.loc['MSFT', 'Price']     -> <class 'numpy.float64'>
sp.loc['MSFT']              -> <class 'pandas.core.series.Series'>
sp.loc[['MSFT'], ['Price']] -> <class 'pandas.core.frame.DataFrame'>


In [44]:
# 3
sp.iloc[10:20, 1:4]

,Sector,Price,Dividend Yield
Symbol,,,
A,Health Care,56.18,0.91
GAS,Utilities,52.98,3.61
APD,Materials,118.74,2.45
ARG,Materials,105.66,1.83
AKAM,Information Technology,53.65,NaN
AA,Materials,13.52,0.91
ALXN,Health Care,162.30,NaN
ATI,Materials,40.99,1.33
ALLE,Industrials,52.46,0.16


In [45]:
# 4
sp.loc['ABT':'ACN', ['Name', 'Price']]

,Name,Price
Symbol,,
ABT,Abbott Laboratories,39.60
ABBV,AbbVie Inc.,53.95
ACN,Accenture,79.79


In [46]:
# 5
sp.iloc[-1:-6:-1]

,Name,Sector,Price,Dividend Yield,Market Cap
Symbol,,,,,
ZTS,Zoetis Inc,Health Care,30.53,0.90,15.296
ZION,Zions Bancorp,Financials,28.43,0.56,5.257
ZMH,Zimmer Holdings,Health Care,101.84,0.81,17.091
YUM,Yum! Brands Inc,Consumer Discretionary,74.77,1.93,33.002
YHOO,Yahoo Inc.,Information Technology,35.02,NaN,35.258


**Объяснение к п. 6:** В `iloc` срез работает по числовым позициям, как в обычном Python — правая граница **не включается**, поэтому `0:3` возвращает строки с позициями 0, 1 и 2 (3 строки). В `loc` срез работает по меткам, и правая граница **включается**, поэтому строка с меткой `ACN` попадает в результат.

### Задание 4. Фильтрация
*Раздел 8*
1. Сохраните среднюю цену всех компаний в переменную `avg_price`. С помощью логической маски с `&` найдите компании сектора `Information Technology` с ценой выше `avg_price`. Сколько их (`sum()` маски или `len()` результата)?
2. Выполните тот же отбор через `query()`, передав среднюю цену как `@avg_price`. Убедитесь, что результаты совпадают (`equals()`).
3. С помощью `isin()` выберите компании секторов `Energy`, `Utilities`, `Materials`. Сколько их всего? Сколько в каждом из трёх секторов (отдельная маска `==` для каждого)?
4. С помощью `~` и `isin()` выберите компании, **не** относящиеся к секторам `Financials` и `Health Care`. Сколько их?
5. Ответьте с помощью `all()`, `any()`, `sum()`: все ли цены больше 0? есть ли компании с ценой выше 500? у скольких компаний `Dividend Yield` больше 3?
6. С помощью `where()` замените в столбце `Price` все цены меньше 50 на 0 и выведите первые 10 значений.

In [47]:
# 1
avg_price = sp['Price'].mean()
mask = (sp['Sector'] == 'Information Technology') & (sp['Price'] > avg_price)
res_mask = sp[mask]
print('Найдено компаний (маска):', len(res_mask))
res_mask.head()

Найдено компаний (маска): 13


,Name,Sector,Price,Dividend Yield,Market Cap
Symbol,,,,,
ACN,Accenture,Information Technology,79.79,2.34,50.513
ADS,Alliance Data Systems Corp,Information Technology,248.88,NaN,13.491
AAPL,Apple Inc.,Information Technology,614.13,2.05,529.000
ADP,Automatic Data Processing,Information Technology,78.85,2.34,38.014
FFIV,F5 Networks,Information Technology,107.75,NaN,8.159


In [48]:
# 2
res_query = sp.query("Sector == 'Information Technology' and Price > @avg_price")
print('Результаты совпадают:', res_mask.equals(res_query))

Результаты совпадают: True


In [49]:
# 3
mask_3 = sp['Sector'].isin(['Energy', 'Utilities', 'Materials'])
print('Всего в трёх секторах:', mask_3.sum())
print('Energy   :', (sp['Sector'] == 'Energy').sum())
print('Utilities:', (sp['Sector'] == 'Utilities').sum())
print('Materials:', (sp['Sector'] == 'Materials').sum())

Всего в трёх секторах: 103
Energy   : 41
Utilities: 33
Materials: 29


In [50]:
# 4
mask_not = ~sp['Sector'].isin(['Financials', 'Health Care'])
print('Компаний не из Financials и Health Care:', mask_not.sum())

Компаний не из Financials и Health Care: 364


In [51]:
# 5
print('Все цены > 0:', (sp['Price'] > 0).all())
print('Есть цены > 500:', (sp['Price'] > 500).any())
print('Dividend Yield > 3:', (sp['Dividend Yield'] > 3).sum())

Все цены > 0: False
Есть цены > 500: True
Dividend Yield > 3: 91


In [52]:
# 6
sp['Price'].where(sp['Price'] >= 50, other=0).head(10)

Symbol
MMM     141.14
ABT       0.00
ABBV     53.95
ACN      79.79
ACE     102.91
ACT     213.77
ADBE     64.30
AES       0.00
AET      76.39
AFL      61.31
Name: Price, dtype: float64

### Задание 5. Сортировка и изменение структуры
*Разделы 9–10*
Работайте с копией: `sp2 = sp.copy()`.
1. С помощью `assign()` добавьте столбец `Dividend` = `Price * Dividend Yield / 100` (дивиденд на акцию).
2. Оператором `[]` добавьте логический столбец `Is_Expensive` = цена выше медианной цены.
3. Методом `insert()` вставьте на позицию 1 столбец `Price_rounded` с округлённой ценой.
4. Переименуйте `Dividend Yield` в `DivYield`, `Market Cap` — в `MarketCap`. Удалите столбец `Name` методом `drop()`. Извлеките столбец `Is_Expensive` методом `pop()` и посчитайте, сколько в нём значений `True`.
5. Отсортируйте `sp2` по `Sector` по алфавиту, а внутри сектора — по `Dividend` по убыванию; выведите первые 10 строк.
6. Выведите 3 компании с наибольшим `Dividend` (`nlargest()`) и 3 самые дешёвые акции (`nsmallest()`). Отсортируйте `sp2` по индексу в обратном порядке (`sort_index()`).

In [53]:
sp2 = sp.copy()

# 1
sp2 = sp2.assign(Dividend=sp2['Price'] * sp2['Dividend Yield'] / 100)

# 2
sp2['Is_Expensive'] = sp2['Price'] > sp2['Price'].median()

# 3
sp2.insert(1, 'Price_rounded', sp2['Price'].round())

# 4
sp2 = sp2.rename(columns={'Dividend Yield': 'DivYield', 'Market Cap': 'MarketCap'})
sp2 = sp2.drop(columns=['Name'])
is_exp = sp2.pop('Is_Expensive')
print('Количество True в Is_Expensive:', is_exp.sum())
sp2.head(3)

Количество True в Is_Expensive: 250


,Price_rounded,Sector,Price,DivYield,MarketCap,Dividend
Symbol,,,,,,
MMM,141.0,Industrials,141.14,2.12,92.345,2.992168
ABT,40.0,Health Care,39.60,1.82,59.477,0.720720
ABBV,54.0,Health Care,53.95,3.02,85.784,1.629290


In [54]:
# 5
sp2.sort_values(by=['Sector', 'Dividend'], ascending=[True, False]).head(10)

,Price_rounded,Sector,Price,DivYield,MarketCap,Dividend
Symbol,,,,,,
WYNN,208.0,Consumer Discretionary,207.86,3.60,20.957,7.482960
GHC,677.0,Consumer Discretionary,677.29,0.76,NaN,5.147404
MCD,102.0,Consumer Discretionary,102.00,3.08,100.800,3.141600
WHR,147.0,Consumer Discretionary,147.26,1.80,11.448,2.650680
HOT,79.0,Consumer Discretionary,78.73,3.00,15.116,2.361900
LB,57.0,Consumer Discretionary,57.32,3.95,16.680,2.264140
DRI,50.0,Consumer Discretionary,49.55,4.44,6.537,2.200020
GPC,85.0,Consumer Discretionary,85.39,2.57,13.116,2.194523
SHW,202.0,Consumer Discretionary,202.41,1.05,20.005,2.125305


In [55]:
# 6
display(sp2.nlargest(3, 'Dividend'))
display(sp2.nsmallest(3, 'Price'))
display(sp2.sort_index(ascending=False).head())

,Price_rounded,Sector,Price,DivYield,MarketCap,Dividend
Symbol,,,,,,
AAPL,614.0,Information Technology,614.13,2.05,529.000,12.589665
WYNN,208.0,Consumer Discretionary,207.86,3.60,20.957,7.482960
BLK,301.0,Financials,300.69,2.32,50.858,6.976008


,Price_rounded,Sector,Price,DivYield,MarketCap,Dividend
Symbol,,,,,,
BEAM,0.0,Consumer Discretionary,0.00,NaN,NaN,NaN
FTR,6.0,Telecommunications Services,5.81,6.97,5.823,0.404957
SLM,9.0,Financials,8.82,2.42,3.729,0.213444


,Price_rounded,Sector,Price,DivYield,MarketCap,Dividend
Symbol,,,,,,
ZTS,31.0,Health Care,30.53,0.90,15.296,0.274770
ZMH,102.0,Health Care,101.84,0.81,17.091,0.824904
ZION,28.0,Financials,28.43,0.56,5.257,0.159208
YUM,75.0,Consumer Discretionary,74.77,1.93,33.002,1.443061
YHOO,35.0,Information Technology,35.02,NaN,35.258,NaN


### Задание 6. Арифметика и выравнивание
*Разделы 11–12*
1. Создайте Series продаж по магазинам: `jan` с индексом `['A', 'B', 'C', 'D']` и `feb` с индексом `['B', 'C', 'D', 'E']`.
2. Посчитайте сумму продаж за два месяца оператором `+` и методом `add(fill_value=0)`. Объясните, откуда берутся `NaN` в первом случае.
3. Найдите магазины, присутствующие в обоих месяцах (`index.intersection()`), и для них посчитайте прирост продаж в процентах `(feb - jan) / jan * 100`, выбирая данные через `loc`.
4. Приведите `jan` и `feb` к общему индексу `['A', 'B', 'C', 'D', 'E']` методом `reindex(..., fill_value=0)` и сложите. Выведите этот результат и результат `add(fill_value=0)` рядом в одном DataFrame (как в разделе 11.2) и сравните значения.
5. Для `sp` из задания 2 одной операцией DataFrame − Series вычтите из столбцов `Price` и `Market Cap` их средние значения. Для `Price` найдите сумму, среднее и максимум.

In [56]:
# 1
jan = pd.Series([100, 200, 300, 400], index=['A', 'B', 'C', 'D'])
feb = pd.Series([220, 270, 440, 150], index=['B', 'C', 'D', 'E'])

# 2
sum_plus = jan + feb
sum_add = jan.add(feb, fill_value=0)
print('Оператор +:\n', sum_plus)
print('\nМетод add:\n', sum_add)

Оператор +:
 A      NaN
B    420.0
C    570.0
D    840.0
E      NaN
dtype: float64

Метод add:
 A    100.0
B    420.0
C    570.0
D    840.0
E    150.0
dtype: float64


**Объяснение к п. 2:** При сложении оператором `+` pandas автоматически выравнивает Series по меткам индекса. Метки `'A'` нет в `feb`, а метки `'E'` нет в `jan`, поэтому на этих позициях получается `NaN`.

In [57]:
# 3
common = jan.index.intersection(feb.index)
growth = (feb.loc[common] - jan.loc[common]) / jan.loc[common] * 100
print('Прирост продаж, %:\n', growth.round(2))

Прирост продаж, %:
 B    10.0
C   -10.0
D    10.0
dtype: float64


In [58]:
# 4
all_idx = ['A', 'B', 'C', 'D', 'E']
sum_reindex = jan.reindex(all_idx, fill_value=0) + feb.reindex(all_idx, fill_value=0)

pd.DataFrame({'reindex': sum_reindex, 'add(fill_value=0)': sum_add})

,reindex,add(fill_value=0)
A,100,100.0
B,420,420.0
C,570,570.0
D,840,840.0
E,150,150.0


In [59]:
# 5
num_sp = sp[['Price', 'Market Cap']]
display((num_sp - num_sp.mean()).head())

print('Сумма Price   :', round(sp['Price'].sum(), 2))
print('Среднее Price :', round(sp['Price'].mean(), 2))
print('Максимум Price:', sp['Price'].max())

,Price,Market Cap
Symbol,,
MMM,63.7714,56.996475
ABT,-37.7686,24.128475
ABBV,-23.4186,50.435475
ACN,2.4214,15.164475
ACE,25.5414,-0.595525


Сумма Price   : 38684.3
Среднее Price : 77.37
Максимум Price: 1197.12


### Задание 7. Большой датасет и экономия памяти
*Раздел 13*
Работайте с файлом `sales_1m.csv` (1 млн строк).
1. Прочитайте первые 1000 строк (`nrows`). По `dtypes`, `describe()` и `nunique()` решите, какой тип подходит каждому столбцу (`int8`, `int32`, `float32`, `category`). Составьте словарь `dtype`.
2. Загрузите файл: а) полностью без параметров; б) только со столбцами `date`, `category`, `qty`, `price` (`usecols`) и вашим `dtype`. Измерьте время чтения (`time.time()`) и память (`memory_usage(deep=True)`), оформите результат в таблицу DataFrame, как в разделе 13.5.
3. Из полной таблицы отберите заказы категории `'Книги'` с `qty > 10`: а) все столбцы; б) только `qty` и `price`. Для каждого варианта посчитайте число строк, память и долю памяти в процентах от полной таблицы. Объясните, почему память уменьшилась.
4. Не загружая весь файл, прочитайте его кусками по 200 000 строк (`chunksize`, только нужные столбцы и ваш `dtype`). В каждом куске отберите заказы категории `'Книги'` и прибавьте к трём счётчикам: число таких заказов (`len()`), сумму `qty` и сумму `price` (`sum()`). Также запомните максимальную память одного куска.
5. По итоговым счётчикам вычислите среднюю цену книги: **сумма цен / число заказов**. Проверьте результат, посчитав то же самое на полностью загруженных данных (`mean()` для отфильтрованного столбца).
6. Сделайте вывод (3–4 предложения): какие приёмы дали наибольшую экономию памяти и почему.

In [60]:
# 1
sample_1k = pd.read_csv('sales_1m.csv', nrows=1000)
print(sample_1k.dtypes)
display(sample_1k.describe())
print('\nУникальных значений:\n', sample_1k.nunique())

my_dtype = {
    'order_id': 'int32',
    'date': 'category',
    'region': 'category',
    'category': 'category',
    'qty': 'int8',
    'price': 'float32',
    'comment': 'category'
}

,order_id,qty,price
count,1000.000000,1000.000000,1000.000000
mean,499.500000,9.976000,2511.078000
std,288.819436,5.561326,1423.439834
min,0.000000,1.000000,21.210000
25%,249.750000,5.000000,1304.715000
50%,499.500000,10.000000,2530.350000
75%,749.250000,15.000000,3716.190000
max,999.000000,19.000000,4999.810000


order_id      int64
date         object
region       object
category     object
qty           int64
price       float64
comment      object
dtype: object

Уникальных значений:
 order_id    1000
date           1
region         5
category       4
qty           19
price        998
comment        3
dtype: int64


In [61]:
# 2
def mb(d):
    return d.memory_usage(deep=True).sum() / 1024**2

t0 = time.time()
sales_full = pd.read_csv('sales_1m.csv')
t_full = time.time() - t0

t0 = time.time()
sales_opt = pd.read_csv('sales_1m.csv',
                        usecols=['date', 'category', 'qty', 'price'],
                        dtype=my_dtype)
t_opt = time.time() - t0

pd.DataFrame({
    'время, с': [t_full, t_opt],
    'память, МБ': [mb(sales_full), mb(sales_opt)]
}, index=['полная загрузка', 'usecols + dtype']).round(2)

,"время, с","память, МБ"
полная загрузка,1.13,338.64
usecols + dtype,0.97,7.69


In [62]:
# 3
cond = (sales_full['category'] == 'Книги') & (sales_full['qty'] > 10)
f_all = sales_full[cond].copy()
f_two = sales_full.loc[cond, ['qty', 'price']].copy()

rep = pd.DataFrame({
    'строк': [len(sales_full), len(f_all), len(f_two)],
    'МБ': [mb(sales_full), mb(f_all), mb(f_two)]
}, index=['весь датасет', 'фильтр (все столбцы)', 'фильтр (qty и price)']).round(2)
rep['доля памяти, %'] = (rep['МБ'] / rep['МБ'].iloc[0] * 100).round(1)
rep

,строк,МБ,"доля памяти, %"
весь датасет,1000000,338.64,100.0
фильтр (все столбцы),118522,40.46,11.9
фильтр (qty и price),118522,2.71,0.8


**Объяснение к п. 3:** Объём DataFrame в памяти примерно равен `число строк × сумма размеров всех столбцов`. Фильтрация по условию сократила число строк примерно в 8 раз, а выбор только двух числовых столбцов `qty` и `price` дополнительно убрал тяжёлые строковые столбцы.

In [63]:
# 4 и 5
cnt = 0
sum_qty = 0
sum_price = 0.0
max_chunk_mb = 0

for chunk in pd.read_csv('sales_1m.csv',
                         usecols=['category', 'qty', 'price'],
                         dtype=my_dtype,
                         chunksize=200_000):
    max_chunk_mb = max(max_chunk_mb, mb(chunk))
    books = chunk[chunk['category'] == 'Книги']
    cnt += len(books)
    sum_qty += books['qty'].sum()
    sum_price += books['price'].sum()

mean_chunk = sum_price / cnt
mean_full = sales_full.loc[sales_full['category'] == 'Книги', 'price'].mean()

print(f'Макс. память одного куска: {max_chunk_mb:.2f} МБ')
print(f'Заказов книг: {cnt}, сумма qty: {sum_qty}')
print(f'Средняя цена (по кускам): {mean_chunk:.4f}')
print(f'Средняя цена (полностью): {mean_full:.4f}')

del sales_full, sales_opt, f_all, f_two

Макс. память одного куска: 1.15 МБ
Заказов книг: 249971, сумма qty: 2502031
Средняя цена (по кускам): 2505.9902
Средняя цена (полностью): 2505.9902


**Вывод (п. 6):** Наибольшую экономию памяти дал перевод повторяющихся строковых столбцов в тип `category` и отказ от лишних столбцов через `usecols`, поскольку обычные строки занимают больше всего памяти. Уменьшение разрядности числовых типов (`int8`, `int32`, `float32`) и фильтрация ненужных строк дополнительно сократили объём данных. Чтение файла кусками через `chunksize` позволило держать в памяти одновременно только небольшую порцию данных, что необходимо при работе с файлами, превышающими объём ОЗУ.

## 15. Контрольные вопросы
1. Чем Series отличается от DataFrame? Что общего у них с массивом NumPy?
2. Какие свойства есть у объекта Index? Почему он неизменяемый?
3. В чём разница между `loc` и `iloc`? Как они обрабатывают правую границу среза?
4. Когда стоит использовать `at` и `iat` вместо `loc` и `iloc`?
5. Почему условия в логической маске нужно брать в скобки? Чем `&` отличается от `and`?
6. Что делают `isin()` и `query()`? Как передать в `query()` внешнюю переменную?
7. Чем `drop()` отличается от `del` и `pop()`?
8. Что такое выравнивание по индексу? Как избежать `NaN` при сложении Series с разными индексами?
9. Почему `memory_usage()` без `deep=True` занижает объём строковых столбцов?
10. Как `usecols`, `dtype`, фильтрация и `chunksize` помогают работать с большими файлами?